# This is our official file for Bucky Bookroom Dummy Submission
Collaborators: Chloe OH, Lokesh Sai Dasari, Lauren Pratt, Zain Waseem 

In [23]:
import os

# Check your current directory
print(os.getcwd())

# Change to the new directory - modify it to your directory 
os.chdir(r"C:\Users\zwaseem\Badger Scribe")


C:\Users\zwaseem\Badger Scribe


In [55]:
# Step 1: Find and load in the Kaggle dataset files 
from pathlib import Path

INPUT_ROOT = Path(".")  # or Path(".") if you are already inside this folder

for i, path in enumerate(sorted(INPUT_ROOT.rglob("*"))):
    if path.is_file():
        print(path)
        if i >= 4:
            break

images\dominy_001_p002.jpg
images\dominy_001_p003.jpg
images\dominy_002_p002.jpg
images\dominy_002_p003.jpg


In [30]:
# Step 2: Load the metadata and inspect the submission format 

import pandas as pd 
from pathlib import Path 

DATASET_DIR = Path() 
train_df = pd.read_csv(DATASET_DIR/"train.csv")
test_df = pd.read_csv(DATASET_DIR/"test.csv")
sample_submission = pd.read_csv(DATASET_DIR / "sample_submission.csv")

print("Train Shape:", train_df.shape) 
print("Test Shape:", test_df.shape) 

display(train_df.head())
display(test_df.head())
display(sample_submission.head())

print("Train columns:", train_df.columns.tolist())
print("Test columns:", test_df.columns.tolist())
print("Submission columns:", sample_submission.columns.tolist())

Train Shape: (619, 5)
Test Shape: (208, 4)


,page_id,doc_id,text,category,label_source
0,dominy_001_p002,dominy_001,Sag Harbor Jany 17 1834\nMajr Dominy\nDr Sir I...,dominy_accounts,silver_claude
1,dominy_001_p003,dominy_001,Majr F Dominy\nE Hampton\nwith a watch,dominy_accounts,silver_claude
2,dominy_002_p002,dominy_002,WASHINGTON BALL.\nE PLURIBUS UNUM\nThe Manager...,dominy_accounts,human
3,dominy_002_p003,dominy_002,2\nMr N Dominy\nEasthampton\nSAG HARBOR N.Y.\n...,dominy_accounts,silver_claude
4,dominy_007_p002,dominy_007,+ Paid 2 $ 50 Cts\nTo get Fanny Mulford a trun...,dominy_accounts,silver_claude


,page_id,doc_id,category,label_source
0,dominy_003_p002,dominy_003,dominy_accounts,silver_claude
1,dominy_004_p002,dominy_004,dominy_accounts,silver_claude
2,dominy_004_p003,dominy_004,dominy_accounts,silver_claude
3,dominy_005_p002,dominy_005,dominy_accounts,silver_claude
4,dominy_005_p003,dominy_005,dominy_accounts,silver_claude


,page_id,text
0,dominy_003_p002,NaN
1,dominy_004_p002,NaN
2,dominy_004_p003,NaN
3,dominy_005_p002,NaN
4,dominy_005_p003,NaN


Train columns: ['page_id', 'doc_id', 'text', 'category', 'label_source']
Test columns: ['page_id', 'doc_id', 'category', 'label_source']
Submission columns: ['page_id', 'text']


In [31]:
# Step 3: Understand where image IDs map to image files 

from pathlib import Path 

image_files = [ 
    p for p in DATASET_DIR.rglob("*")
    if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".webp"}
]

print(f"Found {len(image_files)} image files")
print(*image_files[:10], sep="\n")

Found 543 image files
images\dominy_001_p002.jpg
images\dominy_001_p003.jpg
images\dominy_002_p002.jpg
images\dominy_002_p003.jpg
images\dominy_003_p002.jpg
images\dominy_004_p002.jpg
images\dominy_004_p003.jpg
images\dominy_005_p002.jpg
images\dominy_005_p003.jpg
images\dominy_006_p002.jpg


In [32]:
# Inspecting metadata 

display(train_df.sample(min(5, len(train_df)), random_state=42))
display(test_df.sample(min(5, len(test_df)), random_state=42))

,page_id,doc_id,text,category,label_source
49,kade_007_p031,kade_007,Norman war Montag\nu. jestern zum Zahnarzt.\nE...,kade_letters,human
583,survey_011_p0014,survey_011,T 7 N. R 5 E.\nNorth Between Secs. 10 & 11\n40...,survey_notes,silver_claude
82,kade_007_p105,kade_007,"das solche große Strecke zwichen\nuns ist, abe...",kade_letters,human
305,kade_128_p093,kade_128,"Ich schicke Dir Dein Gelübte\nmit, so bitte le...",kade_letters,human
109,kade_007_p169,kade_007,"Nun werd Ich wohl\nSchließen müssen, denn\nIch...",kade_letters,human


,page_id,doc_id,category,label_source
161,kade_138_p028,kade_138,kade_letters,human
15,dominy_022_p003,dominy_022,dominy_accounts,silver_claude
73,kade_037_p042,kade_037,kade_letters,human
96,kade_042_p038,kade_042,kade_letters,human
166,kade_138_p039,kade_138,kade_letters,human


In [33]:
# Dummy Submission 

#Define a dummy model and cache 

from hashlib import sha256
from pathlib import Path
import json
from PIL import Image


CACHE_DIR = Path()
CACHE_DIR.mkdir(parents=True, exist_ok=True)


def transcribe(image: Image.Image, meta: dict) -> str:
    """Dummy adapter. Replace this later with a real model."""
    return ""


def cache_key(model_name: str, prompt: str, image_id: str) -> str:
    raw = f"{model_name}|{prompt}|{image_id}".encode("utf-8")
    return sha256(raw).hexdigest()


def cached_transcribe(model_name: str, image: Image.Image, meta: dict) -> str:
    image_id = str(meta["image_id"])
    prompt = meta.get("prompt", "Transcribe this document faithfully.")
    cache_path = CACHE_DIR / f"{cache_key(model_name, prompt, image_id)}.json"

    if cache_path.exists():
        return json.loads(cache_path.read_text())["text"]

    text = transcribe(image, meta)
    cache_path.write_text(json.dumps({"text": text}))
    return text

In [37]:
# Creating a submission pipeline 

import pandas as pd

submission = sample_submission.copy()

# Prefer an ID column that exists in both test and sample submission.
candidate_id_columns = ["page_id", "doc_id", "category", "label_source", "text"]

id_col = next(
    (
        col for col in candidate_id_columns
        if col in test_df.columns and col in submission.columns
    ),
    None,
)

if id_col is None:
    raise ValueError(
        "Could not infer the ID column. "
        f"test columns={test_df.columns.tolist()}, "
        f"submission columns={submission.columns.tolist()}"
    )

prediction_columns = [col for col in submission.columns if col != id_col]

if len(prediction_columns) != 1:
    raise ValueError(
        "Expected exactly one prediction column. "
        f"Found: {prediction_columns}"
    )

prediction_col = prediction_columns[0]

# Critical: use the sample-submission row order.
submission[prediction_col] = ""

print("ID column:", id_col)
print("Prediction column:", prediction_col)
print("Submission shape:", submission.shape)

display(submission.head())

ID column: page_id
Prediction column: text
Submission shape: (208, 2)


,page_id,text
0,dominy_003_p002,
1,dominy_004_p002,
2,dominy_004_p003,
3,dominy_005_p002,
4,dominy_005_p003,


In [49]:
#Validate the submission dataset before submitting 

from pathlib import Path

OUTPUT_PATH = Path("submission.csv")

submission.to_csv(OUTPUT_PATH, index=False)

print(f"Wrote: {OUTPUT_PATH.resolve()}")
print("Rows:", len(submission))
print("Columns:", submission.columns.tolist())

assert len(submission) == len(sample_submission), "Wrong number of rows"
assert submission.columns.tolist() == sample_submission.columns.tolist(), "Wrong columns"
assert submission[id_col].equals(sample_submission[id_col]), "IDs/order do not match sample submission"

print("Success: Submission has the expected schema and ID ordering.")

Wrote: C:\Users\zwaseem\Badger Scribe\submission.csv
Rows: 208
Columns: ['page_id', 'text']
Success: Submission has the expected schema and ID ordering.


SyntaxError: invalid syntax (3214540565.py, line 1)